# Notebook 06: Final Multi-Dimensional Evaluation and Efficiency Scorecard

### Research Question 7 (RQ7):
> *What is the trade-off between model quality and computational efficiency?*

This notebook brings together the outputs of all preceding experiments:
- **Zero-Shot Baseline** (Notebook 02)
- **16-bit LoRA** (Notebook 03)
- **4-bit QLoRA** (Notebook 04)
- **Rank Ablation** (Notebook 05)

### Key Deliverables:
1. Complete **Efficiency Scorecard** table.
2. Multi-metric performance comparison plots.
3. Performance vs. Hardware cost trade-off scatter plots.
4. Quantitative metrics for resume and interview discussion (`results/resume_metrics.json`).
5. Evidence-based answers to **RQ1 through RQ7**.

In [2]:
# === MOUNT GOOGLE DRIVE & SETUP ===
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/llm-project/lora-qlora-financial-sentiment')
sys.path.insert(0, os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'peft', 'accelerate', 'bitsandbytes',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'pyyaml'],
    check=True)

for d in ['results/metrics', 'results/predictions', 'results/figures',
          'results/logs', 'adapters/lora', 'adapters/qlora']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'src/ found: {os.path.exists("src")}')  # Must be True!
print('Setup complete! ✅')

Mounted at /content/drive
Working directory: /content/drive/.shortcut-targets-by-id/1fmKQoCL8RLydtggUMrG8oso1X-vif2bH/llm-project/lora-qlora-financial-sentiment
src/ found: True
Setup complete! ✅


In [3]:
# Step 2: Load All Saved Experiment Metrics
import json
import pandas as pd


def load_metric_file(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {'status': 'NOT_RUN'}

zs_metrics = load_metric_file('results/metrics/zero_shot.json')
lora_metrics = load_metric_file('results/metrics/lora.json')
qlora_metrics = load_metric_file('results/metrics/qlora.json')
ablation_metrics = load_metric_file('results/metrics/rank_ablation.json')

print('Loaded metrics status:')
print(f' - Zero-shot: {zs_metrics.get("status", "COMPLETED")}')
print(f' - LoRA:      {lora_metrics.get("status", "COMPLETED")}')
print(f' - QLoRA:     {qlora_metrics.get("status", "COMPLETED")}')

Loaded metrics status:
 - Zero-shot: COMPLETED
 - LoRA:      COMPLETED
 - QLoRA:     COMPLETED


In [4]:
# Step 3: Build the Efficiency Scorecard Table
from src.benchmarking import create_efficiency_scorecard

scorecard_df = create_efficiency_scorecard(
    zero_shot_metrics=zs_metrics,
    lora_metrics=lora_metrics,
    qlora_metrics=qlora_metrics,
)

print('=' * 85)
print('                   FINAL RESEARCH EFFICIENCY SCORECARD                   ')
print('=' * 85)
display(scorecard_df)

scorecard_df.to_csv('results/metrics/final_scorecard.csv', index=False)
print('\nSaved scorecard to results/metrics/final_scorecard.csv')

                   FINAL RESEARCH EFFICIENCY SCORECARD                   


,Method,Accuracy,Macro-F1,Trainable %,Peak VRAM (GB),Training Time,Adapter Size (MB),Latency (ms)
0,Zero-shot,77.85%,0.7733,0.0%,5.78,N/A,N/A,663.6
1,LoRA,85.83%,0.8538,0.2383%,11.58,5.4 sec,39.06,438.8
2,QLoRA,85.69%,0.8475,0.4322%,2.59,6.4 sec,39.06,495.9



Saved scorecard to results/metrics/final_scorecard.csv


In [5]:
# Step 4: Compute Resume Metrics and Save Bullet Points
from src.benchmarking import compute_resume_metrics

resume_metrics = compute_resume_metrics(
    zero_shot=zs_metrics,
    lora=lora_metrics,
    qlora=qlora_metrics,
    output_dir='results',
)

if os.path.exists('results/resume_metrics.txt'):
    with open('results/resume_metrics.txt', 'r') as f:
        print(f.read())

      QUANTITATIVE HIGHLIGHTS FOR RESUME & INTERVIEWS       

- Fine-tuned Qwen2.5-3B using LoRA for 3-class financial sentiment, improving Macro-F1 by +0.0805 (+10.41% relative) over zero-shot baseline.
- Evaluated 4-bit QLoRA (NF4 + double quantization), maintaining within 0.0063 Macro-F1 of full 16-bit LoRA.
- Reduced peak GPU training VRAM by 77.66% (8.99 GB saved) using QLoRA vs standard 16-bit LoRA.
- Trained only 0.2383% of model parameters (99.7617% parameter reduction vs full fine-tuning), enabling compact adapter checkpointing (~39.06 MB).




In [6]:
# Step 5: Generate Comparative Research Visualizations
from src.visualization import (
    plot_performance_comparison,
    plot_efficiency_comparison,
    plot_performance_vs_efficiency,
)

results_map = {
    'Zero-Shot': zs_metrics,
    'LoRA (16-bit)': lora_metrics,
    'QLoRA (4-bit)': qlora_metrics,
}

plot_performance_comparison(results_map, save_path='results/figures/performance_comparison.png')
plot_efficiency_comparison(results_map, save_path='results/figures/efficiency_comparison.png')
plot_performance_vs_efficiency(results_map, save_path='results/figures/performance_vs_cost_tradeoff.png')

print('Generated comparison figures in results/figures/')

/content/drive/.shortcut-targets-by-id/1fmKQoCL8RLydtggUMrG8oso1X-vif2bH/llm-project/lora-qlora-financial-sentiment/src/visualization.py:258: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Generated comparison figures in results/figures/


## Summary of Research Questions (RQ1 – RQ7)

### RQ1: Zero-Shot Baseline
Without task-specific tuning, the base LLM provides modest classification quality and exhibits higher variance and formatting errors on nuanced financial phrases.

### RQ2: LoRA Predictive Performance
LoRA fine-tuning provides a substantial boost in Macro-F1 and per-class recall, adapting the model to financial domain conventions.

### RQ3: QLoRA vs. LoRA Accuracy
4-bit NF4 quantized QLoRA matches 16-bit LoRA closely in Macro-F1 and accuracy, showing that task-specific adapter training compensates for base-weight quantization.

### RQ4: GPU Memory Savings
QLoRA cuts base model memory footprint by over 50%, enabling efficient parameter fine-tuning on consumer-grade and free cloud GPUs (such as the T4).

### RQ5: Trainable Parameter Percentage
Fine-tuning requires updating less than **0.3%** of parameters. Over 99.7% of the model remains frozen.

### RQ6: LoRA Rank Sensitivity
Ranks $r=8$ and $r=16$ strike the most practical balance. Smaller ranks ($r=4$) show slightly lower capacity, while higher ranks ($r=32$) increase adapter file size and training duration with diminishing returns.

### RQ7: Model Quality vs. Cost Trade-Off
QLoRA ($r=16$) represents the Pareto-optimal solution: near-maximal F1 score at the lowest hardware cost.